# 04 · Desafio de Analytics — Camada Gold

Respostas às 6 perguntas de negócio usando **apenas** as tabelas do Star Schema e `display()`.

> **Data limite (perguntas 5 e 6):** a data de lançamento **mais recente** entre filmes com status *Lançado* e `data_lancamento <= hoje` (ignora datas futuras e não lançados). Os recortes são `[limite − 24 meses, limite]` e `[limite − 60 meses, limite]`.

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
spark.sql(f"USE CATALOG `{dbutils.widgets.get('catalogo')}`")

### 1. Receita total (R$) somada de todos os filmes da base

In [ ]:
display(spark.sql("""
    SELECT CAST(SUM(receita_brl) AS DECIMAL(24,2)) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

### 2. Top 5 filmes com maior popularidade

In [ ]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

### 3. Quantidade de filmes por gênero (do maior para o menor)

In [ ]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

### 4. Top 10 filmes por receita (US$ e R$) com `RANK()`

In [ ]:
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    ) t
    WHERE posicao_ranking <= 10          -- RANK() preserva empates: nenhum filme do top 10 fica de fora
    ORDER BY posicao_ranking, titulo
"""))

### Data limite para as perguntas 5 e 6

In [ ]:
limite = spark.sql("""
    SELECT MAX(data_lancamento) AS limite
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first()["limite"]
print("Data limite superior (lançamento mais recente, sem datas futuras):", limite)

### 5. Ator com mais participações em filmes lançados nos últimos 2 anos

In [ ]:
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator,
           COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes
    FROM gold.dim_movies m
    JOIN gold.bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_people p          ON p.sk_person_id = b.sk_person_id
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -24) AND DATE '{limite}'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC, ator
    LIMIT 10
"""))
# A 1ª linha é a resposta; as demais mostram o contexto (e eventuais empates).

### 6. Produtora com maior lucro nos últimos 5 anos
Lucro somado por produtora considerando apenas filmes lançados no período e com lucro calculável (receita **e** orçamento conhecidos).

In [ ]:
display(spark.sql(f"""
    SELECT c.nome_produtora,
           COUNT(DISTINCT m.sk_movie_id)        AS qtd_filmes_com_lucro,
           SUM(f.lucro_usd)                     AS lucro_total_usd,
           SUM(f.lucro_brl)                     AS lucro_total_brl
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    JOIN gold.bridge_movie_company b    ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.dim_companies c           ON c.sk_company_id = b.sk_company_id
    WHERE m.status_filme = 'Lançado'
      AND m.data_lancamento BETWEEN add_months(DATE '{limite}', -60) AND DATE '{limite}'
      AND f.lucro_usd IS NOT NULL
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 10
"""))
# A 1ª linha é a resposta.